In [ ]:
import numpy as np
import sys
import os

# Add the project root to the path
sys.path.append('/home/justin/code/point-to-pose')

from point2pose.modules.register.svd_register import SVDRegister

def load_meta_data_and_create_register(meta_data_path='/home/justin/code/point-to-pose/debug/pipeline/meta_data/meata_data.npz'):
    """
    Load the saved meta_data.npz file and extract registration data for debugging.
    
    Args:
        meta_data_path (str): Path to the meta_data.npz file
        
    Returns:
        tuple: (reg_key_points, reg_cur3d, register_instance)
    """
    # Load the meta data
    print(f"Loading meta data from: {meta_data_path}")
    meta_data = np.load(meta_data_path)
    
    # Print available keys in the meta data
    print("Available keys in meta_data:")
    for key in meta_data.keys():
        print(f"  - {key}: {meta_data[key].shape if hasattr(meta_data[key], 'shape') else type(meta_data[key])}")
    
    # Extract the registration data
    reg_key_points = meta_data['reg_key_points']
    reg_cur3d = meta_data['reg_cur3d']
    
    print(f"\nExtracted data:")
    print(f"  reg_key_points shape: {reg_key_points.shape}")
    print(f"  reg_cur3d shape: {reg_cur3d.shape}")
    
    # Create a register instance for debugging
    config = {
        'debug_level': 1,
        'debug_dir': '/home/justin/code/point-to-pose/debug/register'
    }
    register_instance = SVDRegister(config)
    
    print(f"\nCreated SVDRegister instance with debug level: {config['debug_level']}")
    
    return reg_key_points, reg_cur3d, register_instance

# Load the data and create register instance
reg_key_points, reg_cur3d, register = load_meta_data_and_create_register()


In [ ]:
def debug_registration(reg_key_points, reg_cur3d, register_instance):
    """
    Debug the registration process by running it and analyzing the results.
    
    Args:
        reg_key_points: Source point cloud (key points)
        reg_cur3d: Target point cloud (current 3D points)
        register_instance: SVDRegister instance
    """
    print("=== Registration Debug ===")
    print(f"Source points (reg_key_points): {reg_key_points.shape}")
    print(f"Target points (reg_cur3d): {reg_cur3d.shape}")
    
    # Check if we have the same number of points
    if reg_key_points.shape[0] != reg_cur3d.shape[0]:
        print(f"WARNING: Mismatch in number of points!")
        print(f"  Source: {reg_key_points.shape[0]} points")
        print(f"  Target: {reg_cur3d.shape[0]} points")
        # Use the minimum number of points
        min_points = min(reg_key_points.shape[0], reg_cur3d.shape[0])
        reg_key_points = reg_key_points[:min_points]
        reg_cur3d = reg_cur3d[:min_points]
        print(f"  Using first {min_points} points from both")
    
    # Perform registration
    print("\nPerforming registration...")
    transformation_matrix, stats = register_instance.register(reg_key_points, reg_cur3d)
    
    print(f"Transformation matrix shape: {transformation_matrix.shape}")
    print(f"Transformation matrix:")
    print(transformation_matrix)
    
    # Analyze residuals
    residuals = stats.get('residuals', [])
    if len(residuals) > 0:
        print(f"\nResidual statistics:")
        print(f"  Mean residual: {np.mean(residuals):.6f}")
        print(f"  Std residual: {np.std(residuals):.6f}")
        print(f"  Min residual: {np.min(residuals):.6f}")
        print(f"  Max residual: {np.max(residuals):.6f}")
        print(f"  Median residual: {np.median(residuals):.6f}")
    
    return transformation_matrix, stats

# Run the debug function
transformation, registration_stats = debug_registration(reg_key_points, reg_cur3d, register)


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def visualize_registration_results(reg_key_points, reg_cur3d, transformation_matrix):
    """
    Visualize the registration results in 3D.
    
    Args:
        reg_key_points: Source point cloud
        reg_cur3d: Target point cloud  
        transformation_matrix: 4x4 transformation matrix
    """
    # Transform the source points using the computed transformation
    from point2pose.utils.transform import transform_pts
    transformed_points = transform_pts(transformation_matrix, reg_key_points)
    
    # Create 3D plot
    fig = plt.figure(figsize=(15, 5))
    
    # Plot 1: Original points
    ax1 = fig.add_subplot(131, projection='3d')
    ax1.scatter(reg_key_points[:, 0], reg_key_points[:, 1], reg_key_points[:, 2], 
               c='blue', label='Source (reg_key_points)', alpha=0.6, s=20)
    ax1.scatter(reg_cur3d[:, 0], reg_cur3d[:, 1], reg_cur3d[:, 2], 
               c='red', label='Target (reg_cur3d)', alpha=0.6, s=20)
    ax1.set_title('Original Points')
    ax1.legend()
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    
    # Plot 2: After transformation
    ax2 = fig.add_subplot(132, projection='3d')
    ax2.scatter(transformed_points[:, 0], transformed_points[:, 1], transformed_points[:, 2], 
               c='green', label='Transformed Source', alpha=0.6, s=20)
    ax2.scatter(reg_cur3d[:, 0], reg_cur3d[:, 1], reg_cur3d[:, 2], 
               c='red', label='Target (reg_cur3d)', alpha=0.6, s=20)
    ax2.set_title('After Registration')
    ax2.legend()
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_zlabel('Z')
    
    # Plot 3: Residuals
    ax3 = fig.add_subplot(133)
    residuals = np.linalg.norm(transformed_points - reg_cur3d, axis=1)
    ax3.hist(residuals, bins=30, alpha=0.7, edgecolor='black')
    ax3.set_title('Residual Distribution')
    ax3.set_xlabel('Residual Distance')
    ax3.set_ylabel('Frequency')
    ax3.axvline(np.mean(residuals), color='red', linestyle='--', label=f'Mean: {np.mean(residuals):.4f}')
    ax3.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Print transformation details
    print("=== Transformation Analysis ===")
    R = transformation_matrix[:3, :3]
    t = transformation_matrix[:3, 3]
    
    print(f"Rotation matrix determinant: {np.linalg.det(R):.6f}")
    print(f"Translation vector: {t}")
    print(f"Translation magnitude: {np.linalg.norm(t):.6f}")
    
    # Check if rotation is valid (orthogonal)
    should_be_identity = R @ R.T
    print(f"R @ R.T (should be identity):")
    print(should_be_identity)
    print(f"Max deviation from identity: {np.max(np.abs(should_be_identity - np.eye(3))):.6f}")

# Visualize the results
visualize_registration_results(reg_key_points, reg_cur3d, transformation)
